# F2 — Preprocesamiento de cotizaciones de Airbnb Buenos Aires

La fase construye una base trazable para comparar precios cotizados por barrio, tipo de alojamiento y capacidad. La fuente se conserva sin cambios y fuera de Git. Los resultados describen anuncios y condiciones solicitadas de estancia; no reservas realizadas, ingresos ni relaciones causales.

El flujo sigue **obtención → exploración → limpieza → transformación → validación**. La elección del precio y las condiciones comparables se justifica mediante conteos, concordancias y cobertura. Los cálculos residen en `src/preprocesamiento.py`, utilizando pandas y NumPy. No se requieren nuevas dependencias.

**Objetivos abordados:** calidad y cobertura (OE1), base analítica trazable (OE2), resúmenes iniciales por grupos (OE3) y sensibilidad de las comparaciones (OE4). Los gráficos y la interpretación completa se desarrollarán en la fase analítica; este notebook no declara completados F3 o F4.

**Ejecución:** seleccionar la `.venv` del proyecto; reiniciar el kernel, ejecutar todas las celdas y guardar sus salidas. Los CSV por anuncio son locales y no se versionan.


## 1. Obtención y entorno reproducible

Se reutiliza el archivo original local, cuya descarga y licencia CC BY 4.0 están documentadas en el README. Se verifica el kernel, cada dependencia declarada, la referencia de Git y la exclusión de los directorios de datos antes de generar resultados. La referencia de Git describe el momento de consulta; guardar las nuevas salidas puede producir cambios posteriores.


In [1]:
from pathlib import Path
import importlib
import json
import platform
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display

candidatas = [Path.cwd().resolve(), Path.cwd().resolve().parent]
RAIZ = next((p for p in candidatas if (p / 'requirements.txt').is_file() and (p / 'src/reconocimiento.py').is_file()), None)
if RAIZ is None:
    raise RuntimeError('Abrir el proyecto desde su raíz o desde F2/.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.reconocimiento import calcular_sha256, cargar_dataset, consultar_git, huella_tabla, verificar_entorno
from src import preprocesamiento as f2
importlib.reload(f2)

entorno, dependencias = verificar_entorno(RAIZ)
display(entorno, dependencias)
valores_entorno = entorno.set_index('componente')['valor']
assert bool(valores_entorno['Kernel en .venv del proyecto']), 'Seleccionar la .venv del proyecto como kernel.'
assert bool(valores_entorno['Python de la serie 3.13']), 'Se requiere Python 3.13, según el entorno declarado.'
assert bool(valores_entorno['pip check correcto']) and dependencias['cumple_declaracion'].all()
git_inicio = consultar_git(RAIZ)
assert git_inicio['disponible'], 'No se pudo consultar Git desde la raíz del proyecto.'
display(pd.DataFrame([git_inicio]))

for ruta_local in ['data/raw/listings.csv.gz', 'data/processed/comprobacion_f2.csv']:
    resultado = subprocess.run(['git', '-C', str(RAIZ), 'check-ignore', '-q', '--', ruta_local], capture_output=True)
    if resultado.returncode != 0:
        raise RuntimeError(f'{ruta_local} no está excluido de Git. Revisar .gitignore antes de continuar.')
rastreados = subprocess.run(['git', '-C', str(RAIZ), 'ls-files', '--', 'data/raw/', 'data/processed/'],
                           capture_output=True, text=True, encoding='utf-8', check=True)
publicados = [p for p in rastreados.stdout.splitlines() if Path(p).name != '.gitkeep']
assert not publicados, 'Hay archivos de datos rastreados por Git; revisar su distribución antes de continuar.'


,componente,valor
0,Python,3.13.17
1,Intérprete del kernel,c:\Users\devco\Documents\Magister\proyecto-mcd...
2,Plataforma,Windows-11-10.0.26300-SP0
3,Entorno virtual activo,True
4,Kernel en .venv del proyecto,True
5,Python de la serie 3.13,True
6,pip check correcto,True
7,Salida de pip check,No broken requirements found.


,dependencia,declaracion,version_instalada,importacion_correcta,version_fijada,cumple_declaracion,observacion
0,ipykernel,ipykernel==7.4.0,7.4.0,True,True,True,Versión fijada
1,jupyterlab,jupyterlab==4.6.6,4.6.6,True,True,True,Versión fijada
2,notebook,notebook==7.6.5,7.6.5,True,True,True,Versión fijada
3,numpy,numpy==2.5.4,2.5.4,True,True,True,Versión fijada
4,pandas,pandas==3.1.0,3.1.0,True,True,True,Versión fijada


,disponible,rama,commit,cambios_sin_confirmar
0,True,feature/f2-preprocesamiento,3a004826bc9765fcc5c9e88e9384faba9af2d6ba,True


In [2]:
RUTA_ORIGINAL = RAIZ / 'data/raw/listings.csv.gz'
sha_antes = calcular_sha256(RUTA_ORIGINAL)
assert sha_antes == f2.SHA256_ESPERADO, 'El archivo no coincide con la fuente documentada.'
datos = cargar_dataset(RUTA_ORIGINAL)
huella_antes = huella_tabla(datos)
display(pd.DataFrame([{'filas_originales': len(datos), 'columnas_originales': datos.shape[1],
                       'duplicados_exactos': int(datos.duplicated().sum()), 'sha256': sha_antes}]))


,filas_originales,columnas_originales,duplicados_exactos,sha256
0,29753,90,0,c8db6461c06c0de3efbdda6821d6d28c9aef7dff3c6fde...


## 2. Exploración: faltantes y comparación de precios

Se cuantifican los faltantes de las variables inspeccionadas y se valida el formato de `price` antes de convertirlo en una vista auxiliar. La comparación con `price_quote_price_per_night` usa **tolerancia absoluta de 0,01 ARS y tolerancia relativa cero**. Se informa cuántos anuncios tienen cada campo, cuántos tienen ambos y cuántos coinciden.

La moneda y las condiciones solicitadas se contrastan con `quote.currency`, `quote.price_per_night` y las fechas `requested_*` del JSON. El símbolo `$` no determina la moneda. Las fechas de estancia no son fechas de transacciones.


In [3]:
perfil = f2.perfil_seleccion(datos)
vista = f2.preparar_vista(datos)
comparacion = f2.comparar_precios(datos, vista)
display(perfil, comparacion)
display(vista.groupby(['estado_json', 'moneda'], dropna=False).size().reset_index(name='n_anuncios'))


,variable,tipo_original,n_faltantes,porcentaje_faltantes,n_distintos
0,id,int64,0,0.000000,29753
1,host_id,int64,0,0.000000,14623
2,last_scraped,str,0,0.000000,3
3,neighbourhood_cleansed,str,0,0.000000,47
4,room_type,str,0,0.000000,4
5,accommodates,int64,0,0.000000,16
6,price,str,2119,7.121971,11775
7,price_quote_price_per_night,float64,2120,7.125332,11775
8,price_quote_total_price,float64,2120,7.125332,10072
9,price_quote_checkin_date,str,1390,4.671798,325


,indicador,valor
0,anuncios_totales,29753.0
1,price_presente_original,27634.0
2,price_numerico_finito,27634.0
3,price_presente_no_interpretable,0.0
4,precio_quote_presente_original,27633.0
5,precio_quote_numerico_finito,27633.0
6,ambos_numericos_finitos,27633.0
7,solo_price_finito,1.0
8,solo_precio_quote_finito,0.0
9,coinciden_tolerancia_0_01_ARS,27633.0


,estado_json,moneda,n_anuncios
0,ausente,<NA>,1390
1,objeto_quote,ARS,27760
2,objeto_quote,<NA>,603


### Decisión sobre el precio

Se utiliza **`price_quote_price_per_night`**: es numérico y se contrasta con el importe del JSON. Se exige un valor positivo y finito, moneda ARS y concordancia con `quote.price_per_night`. `price` se conserva como contraste diagnóstico. No se imputan importes ni se rescatan precios sin moneda verificable.

## 3. Limpieza: elegibilidad y alternativas para faltantes

Los `id` repetidos detienen el flujo: no se elige arbitrariamente un anuncio. Las reglas excluyen registros sin precio, moneda o condiciones verificables, capacidad entera positiva o categorías principales reconocidas. Se muestran motivos superpuestos y pérdidas secuenciales para evitar sumar dos veces una exclusión.

Para dormitorios y camas se comparan casos completos, media, mediana global y mediana por tipo. Se mide el efecto sobre cobertura, valores fraccionarios y desviación poblacional. La base conserva sus faltantes con indicadores: estas variables complementarias no determinan las comparaciones principales y su imputación introduciría valores no observados.


In [4]:
exclusiones = f2.contar_exclusiones(vista)
alternativas = f2.alternativas_faltantes(vista)
base, cambios_complementarias = f2.preparar_base(vista)
display(exclusiones, alternativas, cambios_complementarias)
assert int(exclusiones['n_excluidos_en_este_paso'].sum()) + len(base) == len(datos)


,regla,n_con_motivo_en_fuente,n_excluidos_en_este_paso,n_restantes
0,id_ausente,0,0,29753
1,moneda_no_ARS,1993,1993,27760
2,precio_no_positivo_o_no_finito,2120,127,27633
3,precio_no_coincide_JSON,2120,0,27633
4,fechas_no_validas,1390,0,27633
5,fechas_no_coinciden_JSON,1390,0,27633
6,capacidad_no_entera_positiva,0,0,27633
7,barrio_ausente,0,0,27633
8,tipo_no_reconocido,0,0,27633


,variable,alternativa,n_original,n_ausentes_o_invalidos,n_resultante,n_rellenos,media_resultante,mediana_resultante,desviacion_poblacional,cambio_porcentual_desviacion,n_valores_fraccionarios
0,dormitorios,casos_completos,27633,13,27620,0,1.153910,1.0,0.949956,0.000000,0
1,dormitorios,media_global,27633,13,27633,13,1.153910,1.0,0.949732,-0.023525,13
2,dormitorios,mediana_global,27633,13,27633,13,1.153838,1.0,0.949738,-0.022908,0
3,dormitorios,mediana_por_tipo_con_respaldo_global,27633,13,27633,13,1.153838,1.0,0.949738,-0.022908,0
4,camas,casos_completos,27633,478,27155,0,1.942773,1.0,1.598253,0.000000,0
5,camas,media_global,27633,478,27633,478,1.942773,1.0,1.584369,-0.868681,478
6,camas,mediana_global,27633,478,27633,478,1.926465,1.0,1.589130,-0.570794,0
7,camas,mediana_por_tipo_con_respaldo_global,27633,478,27633,478,1.942641,1.0,1.584701,-0.847949,0


,variable,faltantes_antes,invalidos_convertidos_a_faltante
0,dormitorios,13,0
1,camas,478,0
2,valoracion,3077,0


## 4. Condiciones comparables y cobertura

La cohorte principal se elige por la **mayor cantidad de anuncios con idéntica fecha de entrada y duración**, sin ordenar ni seleccionar por el nivel de precio. Los empates se resuelven por fecha más temprana y duración menor. Se conserva toda la tabla de cobertura y se mide la composición por barrio, tipo y capacidad.

La sensibilidad amplía la entrada al mismo mes, conservando la duración. Esto permite comparar resultados al ampliar cobertura, pero no constituye un ajuste por inflación ni garantiza equivalencia económica. Las diferencias de composición limitan la generalización.


In [5]:
principal, sensibilidad, ventanas, decision = f2.seleccionar_cohortes(base)
composicion = f2.medir_composicion(base, principal)
display(pd.DataFrame([decision]), ventanas.head(15))
display(composicion.loc[composicion['variable'].eq('tipo_alojamiento')])


,regla,fecha_entrada_principal,noches,n_base_elegible,n_principal,porcentaje_base_en_principal,mes_entrada_sensibilidad,n_sensibilidad,limitacion
0,Mayor cobertura por fecha exacta de entrada y ...,2026-09-27,1,27633,2080,7.527232,2026-09,5039,La cohorte no es representativa de toda la ofe...


,entrada,noches,n_anuncios
0,2026-09-27,1,2080
1,2026-09-28,2,1279
2,2026-09-27,2,1244
3,2026-09-28,1,1057
4,2026-09-26,1,894
5,2026-09-27,3,640
6,2026-09-29,2,618
7,2026-09-29,1,526
8,2026-09-26,2,521
9,2026-09-28,3,521


,variable,grupo,n_base,n_principal,porcentaje_base,porcentaje_principal,retencion_del_grupo_pct
47,tipo_alojamiento,Entire home/apt,24919,1692,90.178410,81.346154,6.790000
48,tipo_alojamiento,Private room,2437,329,8.819165,15.817308,13.500205
49,tipo_alojamiento,Shared room,151,19,0.546448,0.913462,12.582781
50,tipo_alojamiento,Hotel room,126,40,0.455977,1.923077,31.746032


## 5. Transformación y revisión de atípicos

Se mantienen el precio en ARS y las etiquetas de barrio y tipo. `precio_log1p` facilita inspeccionar asimetrías; su reversibilidad se comprueba. `precio_ars_por_plaza` relaciona tarifa y capacidad anunciada y no representa un cobro por huésped.

El tipo de alojamiento se codifica adicionalmente como cuatro indicadores one-hot, sin imponer orden. Su suma por fila debe ser uno; sus totales verifican la cobertura por categoría. No se declara una variable ordinal inexistente.

Los atípicos de precio se marcan mediante 1,5 IQR dentro del tipo de alojamiento. Se conservan en la base: una tarifa alta no acredita un error. Se compara la distribución con y sin marcas, como análisis de sensibilidad.


In [6]:
indicadores_tipo = f2.codificar_tipos(base)
atipicos = base.groupby('tipo_alojamiento')['precio_atipico_iqr_tipo'].agg(n='size', n_marcados='sum').reset_index()
display(indicadores_tipo.drop(columns='id').sum().rename('n_anuncios').to_frame(), atipicos)
sensibilidad_atipicos = []
for nombre, seleccion in [('principal_completa', principal),
                          ('principal_sin_marcas_IQR', principal.loc[~principal['precio_atipico_iqr_tipo']])]:
    sensibilidad_atipicos.append({'alternativa': nombre, 'n': len(seleccion),
                                 'mediana_ARS': seleccion['precio_ars_noche'].median(),
                                 'q1_ARS': seleccion['precio_ars_noche'].quantile(.25),
                                 'q3_ARS': seleccion['precio_ars_noche'].quantile(.75)})
sensibilidad_atipicos = pd.DataFrame(sensibilidad_atipicos)
display(sensibilidad_atipicos)


,n_anuncios
tipo_Entire home/apt,24919
tipo_Private room,2437
tipo_Shared room,151
tipo_Hotel room,126


,tipo_alojamiento,n,n_marcados
0,Entire home/apt,24919,2004
1,Hotel room,126,2
2,Private room,2437,196
3,Shared room,151,7


,alternativa,n,mediana_ARS,q1_ARS,q3_ARS
0,principal_completa,2080,114662.0,90811.5975,160526.0
1,principal_sin_marcas_IQR,1902,110075.0,87142.5800,145238.0


### Resúmenes iniciales y sensibilidad temporal

Se informan n, mediana y cuartiles para cada barrio, tipo y capacidad de la cohorte principal. Los grupos con menos de 30 anuncios se señalan como pequeños; ese umbral es una advertencia descriptiva, no una garantía de precisión para los demás grupos. Las comparaciones por tipo reúnen capacidades y barrios diferentes: su interpretación requiere considerar esa composición.


In [7]:
barrios = f2.resumir_grupos(principal, 'barrio')
tipos = f2.resumir_grupos(principal, 'tipo_alojamiento')
capacidades = f2.resumir_grupos(principal, 'capacidad')
sensibilidad_tipos = tipos.merge(f2.resumir_grupos(sensibilidad, 'tipo_alojamiento'),
                                 on='tipo_alojamiento', how='outer', suffixes=('_exacta', '_mismo_mes'))
display(tipos, capacidades, barrios, sensibilidad_tipos)


,tipo_alojamiento,n,mediana_ARS,q1_ARS,q3_ARS,grupo_menor_30
0,Entire home/apt,1692,120800.0,98302.9475,166640.85,False
1,Hotel room,40,253784.0,199257.0000,298822.25,False
2,Private room,329,69000.0,47000.0000,107018.00,False
3,Shared room,19,35775.0,24431.0000,42654.00,True


,capacidad,n,mediana_ARS,q1_ARS,q3_ARS,grupo_menor_30
0,1,119,48157.660,35774.6300,68657.5000,False
1,2,909,106252.630,86455.0000,137594.0000,False
2,3,388,114646.265,94786.8775,154353.0725,False
3,4,456,135285.345,105489.0000,184986.6850,False
4,5,86,164131.500,121479.5000,228023.3075,False
5,6,60,195229.995,152845.8200,243291.9575,False
6,7,12,195069.105,115521.2500,291040.5000,True
7,8,20,226257.005,144473.7500,340162.0000,True
8,9,7,305762.930,231034.4700,337104.0000,True
9,10,5,262436.320,224429.9900,359845.1500,True


,barrio,n,mediana_ARS,q1_ARS,q3_ARS,grupo_menor_30
0,Almagro,46,104724.500,61076.2950,1.253632e+05,False
1,Balvanera,78,87143.000,67382.7500,1.177190e+05,False
2,Barracas,15,95681.000,82333.5000,1.185525e+05,True
3,Belgrano,171,140650.950,101666.2900,1.779540e+05,False
4,Boca,5,128421.000,99373.0000,1.635840e+05,True
5,Boedo,2,50422.200,35913.1000,6.493130e+04,True
6,Caballito,17,102125.000,44336.0000,1.161900e+05,True
7,Chacarita,57,96316.000,79499.0000,1.367910e+05,False
8,Coghlan,7,92951.920,91767.2050,1.413450e+05,True
9,Colegiales,43,137440.440,93258.5000,1.872800e+05,False


,tipo_alojamiento,n_exacta,mediana_ARS_exacta,q1_ARS_exacta,q3_ARS_exacta,grupo_menor_30_exacta,n_mismo_mes,mediana_ARS_mismo_mes,q1_ARS_mismo_mes,q3_ARS_mismo_mes,grupo_menor_30_mismo_mes
0,Entire home/apt,1692,120800.0,98302.9475,166640.85,False,3981,125000.0,99110.000,175814.0000,False
1,Hotel room,40,253784.0,199257.0000,298822.25,False,105,253784.0,195171.000,316465.0000,False
2,Private room,329,69000.0,47000.0000,107018.00,False,856,70326.0,47355.195,115516.5575,False
3,Shared room,19,35775.0,24431.0000,42654.00,True,97,32106.0,26651.390,50451.0000,False


## 6. Validación, pruebas y exportación

Se comprueban identificadores, precios, moneda, fechas, capacidad, categorías y transformaciones. Se cotejan las huellas del archivo y de la tabla original antes y después. Cada CSV por anuncio se exporta localmente y se relee para verificar valores y estructura. Estos archivos contienen identificadores de trazabilidad y permanecen fuera de Git.

La suite incluye los ocho casos de F1 y diez casos de F2 sobre entradas normales, límites y errores. Su salida se conserva como evidencia; el flujo se detiene si una prueba falla.


In [8]:
validacion = f2.validar_base(base)
assert calcular_sha256(RUTA_ORIGINAL) == sha_antes, 'El archivo original cambió.'
assert huella_tabla(datos) == huella_antes, 'La tabla original cambió.'
local = RAIZ / 'data/processed'
verificaciones_exportacion = []
for nombre, tabla in [('base_cotizaciones_f2', base), ('cohorte_principal_f2', principal),
                      ('cohorte_sensibilidad_f2', sensibilidad)]:
    control = f2.exportar_y_verificar(tabla, local / f'{nombre}.csv')
    control.insert(0, 'archivo_local', f'data/processed/{nombre}.csv')
    verificaciones_exportacion.append(control)
indicadores_tipo.to_csv(local / 'indicadores_tipo_f2.csv', index=False, encoding='utf-8')
indicadores_releidos = pd.read_csv(local / 'indicadores_tipo_f2.csv', dtype={'id': 'string'})
pd.testing.assert_frame_equal(indicadores_tipo, indicadores_releidos, check_dtype=False)
validacion = pd.concat([validacion, pd.DataFrame([
    {'control': 'archivo_original_inmutable', 'cumple': True},
    {'control': 'tabla_original_inmutable', 'cumple': True},
    {'control': 'tres_CSV_releidos_y_cotejados', 'cumple': True},
    {'control': 'indicadores_tipo_releidos_y_cotejados', 'cumple': True},
])], ignore_index=True)
display(validacion, pd.concat(verificaciones_exportacion, ignore_index=True))


,control,cumple
0,base_no_vacia,True
1,id_unico_y_presente,True
2,precio_finito_positivo,True
3,moneda_ARS,True
4,fechas_y_duracion_coherentes,True
5,capacidad_entera_positiva,True
6,grupos_principales_presentes,True
7,tipos_reconocidos,True
8,log_precio_reversible,True
9,razon_por_plaza_coherente,True


,archivo_local,control,cumple,filas,columnas
0,data/processed/base_cotizaciones_f2.csv,exportacion_y_relectura,True,27633,21
1,data/processed/cohorte_principal_f2.csv,exportacion_y_relectura,True,2080,21
2,data/processed/cohorte_sensibilidad_f2.csv,exportacion_y_relectura,True,5039,21


In [9]:
carpeta_evidencias = RAIZ / 'docs/evidencias'
carpeta_evidencias.mkdir(parents=True, exist_ok=True)
pruebas = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
                         cwd=RAIZ, capture_output=True, text=True, encoding='utf-8', errors='replace')
salida_pruebas = pruebas.stdout + pruebas.stderr
(carpeta_evidencias / 'pruebas_preprocesamiento_f2.txt').write_text(salida_pruebas, encoding='utf-8')
print(salida_pruebas)
assert pruebas.returncode == 0, 'La suite falló; revisar la evidencia antes de continuar.'


test_complementarias_totalmente_nulas_y_fuera_de_rango (test_preprocesamiento.PreprocesamientoTests.test_complementarias_totalmente_nulas_y_fuera_de_rango) ... ok
test_desempate_no_depende_de_precio_ni_orden (test_preprocesamiento.PreprocesamientoTests.test_desempate_no_depende_de_precio_ni_orden) ... ok
test_dominios_y_categorias_inesperadas (test_preprocesamiento.PreprocesamientoTests.test_dominios_y_categorias_inesperadas) ... ok
test_flujo_no_muta_y_cuadra_cohorte_exportacion (test_preprocesamiento.PreprocesamientoTests.test_flujo_no_muta_y_cuadra_cohorte_exportacion) ... ok
test_id_duplicado_columnas_ausentes_y_cero_filas_se_rechazan (test_preprocesamiento.PreprocesamientoTests.test_id_duplicado_columnas_ausentes_y_cero_filas_se_rechazan) ... ok
test_moneda_json_fechas_y_precio_invalidos_excluyen_con_conteos (test_preprocesamiento.PreprocesamientoTests.test_moneda_json_fechas_y_precio_invalidos_excluyen_con_conteos) ... ok
test_precio_rechaza_separadores_ambiguos_y_no_oculta_difer

## 7. Documentación de la ejecución

Se guardan las tablas agregadas, el registro técnico de decisiones y un JSON de procedencia con versiones, referencia Git, controles y huellas de los resultados locales. Ninguno de estos informes publica nombres de anfitriones, enlaces de perfiles ni filas del archivo original.

La tabla de faltantes y la relación objetivos–evidencias alimentan el informe sumativo. Este registro no sustituye el informe institucional ni completa por sí solo las fases posteriores.


In [10]:
tablas = {
    'faltantes': perfil, 'comparacion_precios': comparacion, 'exclusiones': exclusiones,
    'alternativas_faltantes': alternativas, 'cambios_complementarias': cambios_complementarias,
    'ventanas': ventanas, 'composicion': composicion, 'barrios': barrios, 'tipos': tipos,
    'capacidades': capacidades, 'atipicos': atipicos, 'sensibilidad_atipicos': sensibilidad_atipicos,
    'sensibilidad_tipos': sensibilidad_tipos, 'validacion': validacion,
}
carpeta_tablas = RAIZ / 'docs/datos/f2'
carpeta_tablas.mkdir(parents=True, exist_ok=True)
for nombre, tabla in tablas.items():
    tabla.to_csv(carpeta_tablas / f'{nombre}.csv', index=False, encoding='utf-8')
contexto = {'sha256': sha_antes, 'python': platform.python_version(), 'pandas': pd.__version__,
            'numpy': np.__version__, 'git': git_inicio,
            'ambito': 'Ejecución local del notebook con kernel .venv del proyecto.'}
f2.guardar_informe(RAIZ / 'docs/datos/preprocesamiento_f2.md', tablas, decision, contexto)
evidencia = {
    'contexto': contexto, 'decision_cohorte': decision,
    'filas_originales': len(datos), 'columnas_originales': datos.shape[1],
    'filas_base_elegible': len(base), 'columnas_base_elegible': base.shape[1],
    'pruebas_correctas': pruebas.returncode == 0,
    'controles': json.loads(validacion.to_json(orient='records')),
    'sha256_salidas_locales': {p.name: calcular_sha256(p) for p in sorted(local.glob('*_f2.csv'))},
}
(carpeta_evidencias / 'preprocesamiento_f2.json').write_text(
    json.dumps(evidencia, ensure_ascii=False, indent=2, allow_nan=False) + '\n', encoding='utf-8')
print('Generados: docs/datos/preprocesamiento_f2.md, docs/datos/f2/ y evidencias de F2.')
print('Los CSV por anuncio permanecen en data/processed/, fuera de Git. Guardar este notebook con Ctrl+S.')


Generados: docs/datos/preprocesamiento_f2.md, docs/datos/f2/ y evidencias de F2.
Los CSV por anuncio permanecen en data/processed/, fuera de Git. Guardar este notebook con Ctrl+S.


## Alcance de los resultados

La base conserva cotizaciones verificadas y una cohorte con condiciones iguales de estancia. No representa todas las fechas ni todos los anuncios de Buenos Aires. La disponibilidad de cotización y la elección de una fecha pueden sesgar la composición. Varios anuncios pueden corresponder a un mismo anfitrión; no se presupone independencia estadística entre ellos.

La ventana mensual es una sensibilidad de cobertura, no una corrección inflacionaria. Las medianas y los cuartiles describen oferta cotizada; no identifican efectos causales ni una tarifa comercial óptima. El informe institucional deberá interpretar estos límites y citar las fuentes en APA 7.

**Documentación:** [fuente y licencia](../docs/datos/fuente_dataset.md), [objetivos y alcance](../F1/definicion_problema.md), [decisiones de F2](../docs/datos/preprocesamiento_f2.md).
